# Chapter 5 &mdash; State Names as Compressed History

**Concept 6 of the Chapter 5 decomposition:** *State Names as Compressed History: "Ends with 0101"*

"Ends with 0101": remember only the longest relevant suffix, and introduce states on demand.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Compressed-History-Suffix/Concept-Compressed-History-Suffix.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


For "**ends with 0101**" the state must remember **the longest prefix of the pattern
that is currently a suffix of the input** &mdash; nothing more. That is compressed history:
the input may be a million bits, the state is one of five.

Two habits make the design mechanical:

* **introduce states on demand** &mdash; add a state the moment you need to distinguish;
* **cover both symbols from every state**, including the fall-back edges, which is
  where beginners lose the machine.

## 2. Definitions

### The specification

In [ ]:
PAT = '0101'
def in_L(s): return s.endswith(PAT)

### State = longest prefix of `0101` that is a suffix of the input so far

In [ ]:
def longest_overlap(s, pat):
    for k in range(min(len(s), len(pat)), -1, -1):
        if s.endswith(pat[:k]):
            return k
    return 0

### The DFA, one state per overlap length, all fall-backs written out

In [ ]:
E = md2mc('''DFA
I    : 0 -> S0      !! have '0'
I    : 1 -> I       !! nothing usable
S0   : 0 -> S0      !! still just '0'
S0   : 1 -> S01     !! have '01'
S01  : 0 -> S010    !! have '010'
S01  : 1 -> I       !! '011' -- start over
S010 : 0 -> S0      !! '0100' -- keep the trailing 0
S010 : 1 -> F       !! '0101' -- match!
F    : 0 -> S010    !! '01010' -- suffix '010' survives
F    : 1 -> I
''')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;5.&nbsp;Sliding-Window Conditions: "Every Block of 3 Has Exactly Two 1s"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Sliding-Window/Concept-Sliding-Window.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;7.&nbsp;State Names as Residues: MSB-First "Divisible by 3"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Residue-States-MSB/Concept-Residue-States-MSB.ipynb)&nbsp;&rarr;

---

## 3. Tests

The state name **is** the overlap length &mdash; a checkable invariant.

In [ ]:
tag = {'I': 0, 'S0': 1, 'S01': 2, 'S010': 3, 'F': 4}
from itertools import product
for k in range(11):
    for p in product('01', repeat=k):
        s = ''.join(p)
        assert tag[run_dfa(E, s)] == longest_overlap(s, PAT), s
print("state == longest overlap, verified on all strings up to length 10")

And so the machine is correct.

In [ ]:
assert all(accepts_dfa(E, ''.join(p)) == in_L(''.join(p))
           for k in range(12) for p in product('01', repeat=k))
for s in ['0101', '00101', '01011', '0101 0101'.replace(' ',''), '010', '']:
    print("%-10r ends with 0101? %s" % (s, accepts_dfa(E, s)))

The fall-back edges are the subtle part: `F -0-> S010` keeps the useful suffix.

In [ ]:
print("after '0101' then '0' the input ends '01010'")
print("  longest useful suffix is '010' -> state S010, NOT the start state")
assert run_dfa(E, '01010') == 'S010'
print("Throwing that away would miss '010101'. Check:", accepts_dfa(E, '010101'))
assert accepts_dfa(E, '010101')

Five states, however long the input.

In [ ]:
long_in = '0101' * 150          # 600 symbols
print("|Q| =", len(E["Q"]), " input of length %d handled?" % len(long_in),
      accepts_dfa(E, long_in))
print("(Jove's accepts_dfa recurses per symbol, so Python's recursion limit -- not")
print(" the DFA -- caps the input length. The MACHINE is size 5 either way.)")

## 4. Animation

Trace `010101` and watch the fall-back from `F` land on `S010`, not on `I`.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(E, FuseEdges=True)

## 5. Exercises


1. Redo it for the pattern `0110`. Which fall-backs differ, and why?
2. For a pattern of length $k$ with no self-overlap, how many states?
3. This construction has a name in string matching. Which algorithm?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter5/Concept-Compressed-History-Suffix')